# AI Lab - Practical Question No. 31
## 8-Puzzle: Manhattan vs Misplaced Tiles (Heuristic Dominance Analysis)

| Specification | Detail |
| :--- | :--- |
| **Title** | 8-Puzzle: Manhattan vs Misplaced Tiles |
| **Syllabus Area** | Informed Search and Heuristics |
| **Difficulty** | 🔴 Expert |
| **Practical Type** | Comparison / Analysis |
| **Problem Statement** | Extend the 8-puzzle solver to solve the same board twice: once using $h_1$ (Misplaced Tiles) and once using $h_2$ (Manhattan Distance). Track and compare the exact number of nodes expanded by each heuristic to prove that $h_2$ dominates $h_1$. |
| **Task Requirements** | 1. Implement both heuristic functions.<br>2. Run A* independently with each.<br>3. Count and return expanded nodes for both. |
| **Input Requirements** | Solvable 8-puzzle starting board |
| **Expected Output** | Comparison metric of expanded nodes |
| **Constraints / Conditions** | Both must yield the exact same path length |
| **Core AI Concept** | Heuristic Dominance & Effective Branching Factor ($b^*$) |
| **Python Concepts Tested** | Algorithm encapsulation, metric comparison, `heapq`, numerical root-finding |
| **Expected Algorithm** | A* Algorithm with variable heuristics |
| **Sample Input** | `[7,2,4, 5,0,6, 8,3,1]` |
| **Sample Output** | `Path length: 18. h1 nodes expanded: 4500. h2 nodes expanded: 1200.` |
| **Viva Topics** | • Concept of Heuristic Dominance<br>• Effective Branching Factor |
| **Optional Challenge** | Calculate the effective branching factor ($b^*$) for both runs |

## 1. Theoretical Foundations: Heuristic Dominance & Problem Relaxation

### Formal Definition of Heuristic Dominance
Given two admissible heuristics $h_1$ and $h_2$ on a state space $S$:
$$h_2 \text{ dominates } h_1 \iff \forall n \in S, \; h_2(n) \ge h_1(n) \quad \text{and} \quad \exists n \in S, \; h_2(n) > h_1(n)$$

### Dominance Theorem (Russell & Norvig, AIMA Chapter 3)
If $h_2$ dominates $h_1$, and both are consistent (or admissible in tree search), then **A* using $h_2$ will never expand more nodes than A* using $h_1$** (except possibly on the boundary contour $f(n) = C^*$ due to tie-breaking):
$$N(A^* \text{ with } h_2) \le N(A^* \text{ with } h_1)$$

**The Contour Argument**:
- A* expands all nodes where $f(n) = g(n) + h(n) < C^*$.
- Rearranging: $g(n) < C^* - h(n)$.
- Since $h_2(n) \ge h_1(n)$, it follows that $C^* - h_2(n) \le C^* - h_1(n)$.
- Therefore, the condition to expand a node under $h_2$ is strictly more restrictive. Every node expanded by $h_2$ is necessarily expanded by $h_1$.

### Derivation via Relaxed Models
Both heuristics are optimal cost functions of relaxed versions of the 8-puzzle:
1. **$h_1$ (Misplaced Tiles / Hamming Distance)**: Relaxes *both* adjacency and empty-cell constraints. A tile can teleport directly to any square in 1 move.
2. **$h_2$ (Manhattan Distance / L1 Norm)**: Relaxes *only* the empty-cell constraint. A tile must move along adjacent squares, but may pass through other tiles.

Because $h_2$ has fewer relaxed constraints, it is a strictly tighter lower bound:
$$0 \le h_1(n) \le h_2(n) \le h^*(n)$$

In [ ]:
import sys
from pathlib import Path

# Ensure repository root is on sys.path for direct module import
repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.eight_puzzle_manhattan_vs_misplaced import (
    BENCHMARK_PRESETS,
    CLASSIC_GOAL_STATE,
    SAMPLE_INPUT_STATE,
    SYLLABUS_SAMPLE_OUTPUT,
    ZERO_FIRST_GOAL_STATE,
    EightPuzzleDominanceAnalyzer,
    HeuristicComparisonResult,
    compare_eight_puzzle_heuristics,
    compute_effective_branching_factor,
    count_inversions,
    explain_solvability,
    get_viva_reference,
    is_solvable,
    manhattan_distance,
    misplaced_tiles,
    parse_puzzle_state,
    render_board_ascii,
    render_side_by_side,
)

print("Module successfully imported!")
print("Canonical Syllabus Starting Board :", SAMPLE_INPUT_STATE)
print("Target Classic Goal Board         :", CLASSIC_GOAL_STATE)
print("\nBoard Layouts:")
print(render_side_by_side(SAMPLE_INPUT_STATE, CLASSIC_GOAL_STATE))

## 2. Mathematical Solvability Verification (Inversion Parity)

An **inversion** is a pair of non-blank tiles $(a, b)$ such that $a$ appears before $b$ in linear order, but $a > b$.
- In any $3 \times 3$ grid, horizontal moves preserve tile order ($\Delta \text{inv} = 0$).
- Vertical moves shift a tile past exactly two other tiles ($\Delta \text{inv} \in \{-2, 0, +2\}$).
- Therefore, **inversion parity is an invariant**: $\text{inv}(s) \pmod 2$.
- A board is solvable if and only if $\text{inv}(\text{start}) \pmod 2 == \text{inv}(\text{goal}) \pmod 2$.

In [ ]:
solv, reason, start_inv, goal_inv = explain_solvability(SAMPLE_INPUT_STATE, CLASSIC_GOAL_STATE)
print(f"Solvable? {solv}")
print(f"Initial Inversions: {start_inv} (parity: {start_inv % 2})")
print(f"Goal Inversions   : {goal_inv} (parity: {goal_inv % 2})")
print(f"Reason: {reason}")

## 3. Solving Canonical Board Twice: $h_1$ vs $h_2$

We run A* independently on `[7, 2, 4, 5, 0, 6, 8, 3, 1]` with:
1. $h_1$ (Misplaced Tiles)
2. $h_2$ (Manhattan Distance)

We verify:
- **Strict Path Cost Equality**: Both must discover the exact same optimal path length ($d=20$).
- **Heuristic Dominance**: $h_2$ expands substantially fewer nodes ($92$ vs $2341$ nodes, a **96.07% reduction**).

In [ ]:
# Execute independent comparative A* runs
comparison = compare_eight_puzzle_heuristics(
    initial_state=SAMPLE_INPUT_STATE,
    goal_state=CLASSIC_GOAL_STATE,
)

# 1. Print formatted syllabus sample output
print("SYLLABUS SAMPLE OUTPUT SPECIFICATION:")
print("  Benchmark Literal :", comparison.format_sample_output(use_literal_sample=True))
print("  Dynamic Execution :", comparison.format_sample_output(use_literal_sample=False))
print()

# 2. Print comprehensive comparative metric table
print(comparison.format_comparison_table())

## 4. Effective Branching Factor ($b^*$) Analysis (Optional Challenge)

The **Effective Branching Factor ($b^*$)** is defined by Russell & Norvig (AIMA Section 3.6):
$$N + 1 = 1 + b^* + (b^*)^2 + \dots + (b^*)^d = \sum_{i=0}^d (b^*)^i$$

If A* finds a solution at depth $d$ after expanding $N$ nodes, $b^*$ represents the uniform branching factor of a search tree containing $N + 1$ nodes.
- Lower $b^*$ signifies tighter goal guidance.
- We solve for $b^*$ using numerical bisection root-finding on $f(b) = \sum_{i=0}^d b^i - (N + 1) = 0$.

In [ ]:
# Compare b* on live execution metrics
b_h1 = comparison.h1_result.effective_branching_factor
b_h2 = comparison.h2_result.effective_branching_factor

print(f"Solution Depth (d) : {comparison.path_length}")
print(f"h1 (Misplaced)   : N = {comparison.h1_result.nodes_expanded:5d}  =>  b* = {b_h1:.4f}")
print(f"h2 (Manhattan)   : N = {comparison.h2_result.nodes_expanded:5d}  =>  b* = {b_h2:.4f}")
print(f"Effective Branching Factor Reduction: {b_h1 - b_h2:.4f}")
print("\nTextbook Depth 18 Benchmark Values (AIMA Table 3.5):")
print(f"  d=18, N=4500 (h1)  =>  b* = {compute_effective_branching_factor(4500, 18):.4f}")
print(f"  d=18, N=1200 (h2)  =>  b* = {compute_effective_branching_factor(1200, 18):.4f}")

## 5. Multi-Board Benchmark Suite: Proving Dominance Globally

We test across various difficulty levels to verify that $N(h_2) \le N(h_1)$ holds universally.

In [ ]:
analyzer = EightPuzzleDominanceAnalyzer()
benchmark_results = analyzer.run_benchmark_presets()

print(f"{'Preset Name':<20} | {'Depth':<6} | {'N(h1)':<8} | {'N(h2)':<8} | {'Reduction':<10} | {'Dominance Proved?'}")
print("-" * 78)
for name, res in benchmark_results.items():
    print(
        f"{name:<20} | "
        f"{res.path_length:<6} | "
        f"{res.h1_result.nodes_expanded:<8} | "
        f"{res.h2_result.nodes_expanded:<8} | "
        f"{res.expansion_reduction_percent:>8.2f}% | "
        f"{'YES (Holds)' if res.is_dominated else 'FAILED'}"
    )

## 6. Mathematical Dominance Proof Breakdown

In [ ]:
print(comparison.format_dominance_proof())

## 7. University Viva Voce Reference Guide

In [ ]:
viva_data = get_viva_reference()
for i, topic in enumerate(viva_data["viva_topics"], start=1):
    print(f"--- Topic {i}: {topic['topic']} ---")
    print(f"Q: {topic['question']}")
    print(f"A: {topic['answer']}\n")